# 🦊 Fox-Mecha Complete Super Brain LoRA Trainer & GGUF Exporter
Trains on Kaggle GPU and generates `fox_lora.gguf` in ~1-2 minutes!
Auto-handles torchao compatibility, dataset extraction, and GGUF conversion.

In [ ]:
# 1. Environment Setup
!pip install -q transformers peft datasets accelerate gguf
print("✅ Setup complete!")

In [ ]:
# 2. Unpack Dataset & Train on GPU
import gzip, base64, os, torch
import peft.tuners.lora.torchao
peft.tuners.lora.torchao.is_torchao_available = lambda: False
import peft.import_utils
peft.import_utils.is_torchao_available = lambda: False

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model

print("📦 Unpacking 880 Bengali dialogues...")
b64_str = "..."
with open("fox_bengali_dataset.jsonl", "wb") as f:
    f.write(gzip.decompress(base64.b64decode(b64_str.encode("ascii"))))
print("✅ Dataset extracted!")

model_id = "Qwen/Qwen2.5-0.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"⚡ Device: {device.upper()}")

print("📥 Loading Qwen2.5-0.5B...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
    low_cpu_mem_usage=True
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

dataset = load_dataset("json", data_files="fox_bengali_dataset.jsonl", split="train")

def tokenize_func(example):
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)
    enc = tokenizer(text, truncation=True, max_length=256)
    enc["labels"] = enc["input_ids"].copy()
    return enc

tokenized_dataset = dataset.map(tokenize_func, remove_columns=dataset.column_names)

training_args = TrainingArguments(
    output_dir="./fox_checkpoints",
    per_device_train_batch_size=8 if device == "cuda" else 2,
    gradient_accumulation_steps=2 if device == "cuda" else 4,
    learning_rate=3e-4,
    max_steps=120,
    logging_steps=10,
    save_strategy="no",
    fp16=(device == "cuda"),
    bf16=False,
    optim="adamw_torch",
    report_to="none"
)

trainer = Trainer(
    model=model,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, pad_to_multiple_of=8),
    args=training_args,
)

print("🚀 Training (120 steps on GPU)... Approx 60 seconds!")
trainer.train()

save_dir = "./fox_lora_final"
os.makedirs(save_dir, exist_ok=True)
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print("🎉 LoRA trained and saved successfully!")


In [ ]:
# 3. Convert to GGUF and Download directly
!git clone --depth=1 https://github.com/ggerganov/llama.cpp
!python llama.cpp/convert_lora_to_gguf.py ./fox_lora_final --base Qwen/Qwen2.5-0.5B-Instruct --outfile fox_lora.gguf

import os
from IPython.display import FileLink, display

if os.path.exists("fox_lora.gguf"):
    size_mb = os.path.getsize("fox_lora.gguf") / (1024 * 1024)
    print(f"✅ SUCCESS! Generated fox_lora.gguf ({size_mb:.2f} MB)")
    print("👇 Click below to download:")
    display(FileLink("fox_lora.gguf"))
else:
    print("❌ File not found. Checking directory:")
    !ls -lh
